# Extended Thinking with Tool Use

## Table of contents
- [Setup](#setup)
- [Basic example](#basic-example)
- [Multiple tool calls](#multiple-tool-calls-with-thinking)
- [Preserving thinking blocks](#preserving-thinking-blocks)

This notebook demonstrates how to use Claude's extended thinking feature with tools on Claude Sonnet 5.5. The extended thinking feature allows you to see Claude's step-by-step thinking before it provides a final answer, providing transparency into how it decides which tools to use and how it interprets tool results.

On Claude Sonnet 5.5 thinking is adaptive (`thinking={"type": "adaptive"}`, with `output_config={"effort": ...}` controlling how hard it thinks), and the model may think before its first tool request and again between tool calls after it reads tool results (interleaved thinking). No beta header is needed for this. Thinking blocks carry a signature and are bound to the conversation so far, so earlier assistant content, including `thinking` and `redacted_thinking` blocks, must be passed back unchanged on each follow-up request. For more information on extended thinking, see our [documentation](https://platform.claude.com/docs/en/build-with-claude/thinking).

## Setup

First, let's install the necessary packages and set up our environment.

In [ ]:
%pip install anthropic

In [1]:
import json

import anthropic

# Global variables for model, output limit, and thinking effort
MODEL_NAME = "claude-sonnet-5-5"
MAX_TOKENS = 16000  # thinking tokens count toward max_tokens, so leave headroom
EFFORT = "xhigh"  # how hard Claude thinks: low | medium | high (default) | xhigh | max.
# Adaptive thinking may skip thinking entirely on easy requests at lower effort; "xhigh"
# makes these small examples reliably produce thinking blocks you can inspect.

# Every request below passes thinking={"type": "adaptive", "display": "summarized"}.
# "display" defaults to "omitted" on this model, which returns thinking blocks with
# empty text; "summarized" returns a summary of the reasoning so we can print it.

# Set your API key as an environment variable or directly
# os.environ["ANTHROPIC_API_KEY"] = "your_api_key_here"

# Initialize the client
client = anthropic.Anthropic()


# Helper functions
def print_thinking_response(response):
    """Pretty print a message response with thinking blocks."""
    print("\n==== FULL RESPONSE ====")
    for block in response.content:
        if block.type == "thinking":
            print("\n🧠 THINKING BLOCK:")
            # Show truncated thinking for readability
            print(block.thinking[:500] + "..." if len(block.thinking) > 500 else block.thinking)
            print(f"\n[Signature available: {bool(getattr(block, 'signature', None))}]")
            if hasattr(block, "signature") and block.signature:
                print(f"[Signature (first 50 chars): {block.signature[:50]}...]")
        elif block.type == "redacted_thinking":
            print("\n🔒 REDACTED THINKING BLOCK:")
            print(f"[Data length: {len(block.data) if hasattr(block, 'data') else 'N/A'}]")
        elif block.type == "text":
            print("\n✓ FINAL ANSWER:")
            print(block.text)

    print("\n==== END RESPONSE ====")


def count_tokens(messages, tools=None):
    """Count tokens for a given message list with optional tools."""
    if tools:
        response = client.messages.count_tokens(model=MODEL_NAME, messages=messages, tools=tools)
    else:
        response = client.messages.count_tokens(model=MODEL_NAME, messages=messages)
    return response.input_tokens

## Single tool calls with thinking

This example demonstrates how to combine thinking and make a single tool call, with a mock weather tool.

In [2]:
def tool_use_with_thinking():
    # Define a weather tool
    tools = [
        {
            "name": "weather",
            "description": "Get current weather information for a location.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The location to get weather for.",
                    }
                },
                "required": ["location"],
            },
        }
    ]

    def weather(location):
        # Mock weather data, keyed by bare city name (the model may pass "Paris, FR")
        location = location.split(",")[0].strip()
        weather_data = {
            "New York": {"temperature": 72, "condition": "Sunny"},
            "London": {"temperature": 62, "condition": "Cloudy"},
            "Tokyo": {"temperature": 80, "condition": "Partly cloudy"},
            "Paris": {"temperature": 65, "condition": "Rainy"},
            "Sydney": {"temperature": 85, "condition": "Clear"},
            "Berlin": {"temperature": 60, "condition": "Foggy"},
        }

        return weather_data.get(location, {"error": f"No weather data available for {location}"})

    # A bare "what's the weather in Paris?" usually goes straight to the tool call with no
    # thinking. Giving Claude a small decision that depends on the result makes it reason first.
    user_message = {
        "role": "user",
        "content": "I'm hosting an outdoor dinner in Paris this evening and deciding whether to move it indoors. Work out what you need to check, use the tool, then give me a recommendation with your reasoning.",
    }

    # Initial request with tool use and thinking
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=MAX_TOKENS,
        thinking={"type": "adaptive", "display": "summarized"},
        output_config={"effort": EFFORT},
        tools=tools,
        messages=[user_message],
    )

    # Detailed diagnostic output of initial response
    print("\n=== INITIAL RESPONSE ===")
    print(f"Response ID: {response.id}")
    print(f"Stop reason: {response.stop_reason}")
    print(f"Model: {response.model}")
    print(f"Content blocks: {len(response.content)} blocks")

    for i, block in enumerate(response.content):
        print(f"\nBlock {i + 1}: Type = {block.type}")
        if block.type == "thinking":
            print(f"Thinking content: {block.thinking[:150]}...")
            print(f"Signature available: {bool(getattr(block, 'signature', None))}")
        elif block.type == "text":
            print(f"Text content: {block.text}")
        elif block.type == "tool_use":
            print(f"Tool: {block.name}")
            print(f"Tool input: {block.input}")
            print(f"Tool ID: {block.id}")
    print("=== END INITIAL RESPONSE ===\n")

    # Handle tool use until Claude gives its final answer
    full_conversation = [user_message]

    while response.stop_reason == "tool_use":
        # Keep the whole assistant turn (thinking, redacted_thinking, text and tool_use blocks,
        # in order) for the conversation history. Thinking blocks are signed and bound to the
        # conversation prefix, so pass the turn back exactly as received.
        full_conversation.append({"role": "assistant", "content": response.content})

        # Execute every tool_use block in this turn and answer each one
        tool_results = []
        for tool_use_block in (b for b in response.content if b.type == "tool_use"):
            print("\n=== EXECUTING TOOL ===")
            print(f"Tool name: {tool_use_block.name}")
            print(f"Location to check: {tool_use_block.input['location']}")
            tool_result = weather(tool_use_block.input["location"])
            print(f"Result: {tool_result}")
            print("=== TOOL EXECUTION COMPLETE ===\n")
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use_block.id,
                    "content": json.dumps(tool_result),
                }
            )

        # Add the tool results to the conversation in a single user turn
        full_conversation.append({"role": "user", "content": tool_results})

        # Continue the conversation with the same thinking configuration
        print("\n=== SENDING FOLLOW-UP REQUEST WITH TOOL RESULT ===")
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=MAX_TOKENS,
            thinking={"type": "adaptive", "display": "summarized"},
            output_config={"effort": EFFORT},
            tools=tools,
            messages=full_conversation,
        )
        print(f"Follow-up response received. Stop reason: {response.stop_reason}")

    print_thinking_response(response)


# Run the example
tool_use_with_thinking()


=== INITIAL RESPONSE ===
Response ID: msg_011Cfdj5KspNpopEXyAhbhct
Stop reason: tool_use
Model: claude-sonnet-5-5
Content blocks: 3 blocks

Block 1: Type = thinking
Thinking content: I need to check the weather in Paris, so I'll call the weather tool for that.

...
Signature available: True

Block 2: Type = text
Text content: I need to check Paris's current conditions and anything that affects this evening, such as rain, temperature, wind, and how the weather is trending. I'll pull the weather now.

Block 3: Type = tool_use
Tool: weather
Tool input: {'location': 'Paris, France'}
Tool ID: toolu_014FismJG9gGgQfUSw9WVjGs
=== END INITIAL RESPONSE ===


=== EXECUTING TOOL ===
Tool name: weather
Location to check: Paris, France
Result: {'temperature': 65, 'condition': 'Rainy'}
=== TOOL EXECUTION COMPLETE ===


=== SENDING FOLLOW-UP REQUEST WITH TOOL RESULT ===
Follow-up response received. Stop reason: end_turn

==== FULL RESPONSE ====

🧠 THINKING BLOCK:
The data I got back is pretty sparse—

## Multiple tool calls with thinking

This example demonstrates how to handle multiple tool calls, such as a mock news and weather service, while observing the thinking process.

In [3]:
def multiple_tool_calls_with_thinking():
    # Define tools
    tools = [
        {
            "name": "weather",
            "description": "Get current weather information for a location.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The location to get weather for.",
                    }
                },
                "required": ["location"],
            },
        },
        {
            "name": "news",
            "description": "Get latest news headlines for a topic.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string", "description": "The topic to get news about."}
                },
                "required": ["topic"],
            },
        },
    ]

    def weather(location):
        # Mock weather data, keyed by bare city name (the model may pass "Paris, FR")
        location = location.split(",")[0].strip()
        weather_data = {
            "New York": {"temperature": 72, "condition": "Sunny"},
            "London": {"temperature": 62, "condition": "Cloudy"},
            "Tokyo": {"temperature": 80, "condition": "Partly cloudy"},
            "Paris": {"temperature": 65, "condition": "Rainy"},
            "Sydney": {"temperature": 85, "condition": "Clear"},
            "Berlin": {"temperature": 60, "condition": "Foggy"},
        }

        return weather_data.get(location, {"error": f"No weather data available for {location}"})

    def news(topic):
        # Mock news data
        news_data = {
            "technology": [
                "New AI breakthrough announced by research lab",
                "Tech company releases latest smartphone model",
                "Quantum computing reaches milestone achievement",
            ],
            "sports": [
                "Local team wins championship game",
                "Star player signs record-breaking contract",
                "Olympic committee announces host city for 2036",
            ],
            "weather": [
                "Storm system developing in the Atlantic",
                "Record temperatures recorded across Europe",
                "Climate scientists release new research findings",
            ],
        }

        # Loose matching so "London tech" or "technology news" resolve to a category
        topic = topic.lower()
        for key, headlines in news_data.items():
            if key in topic or (key == "technology" and "tech" in topic):
                return {"headlines": headlines}
        return {"headlines": ["No news available for this topic"]}

    # Initial request
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=MAX_TOKENS,
        thinking={"type": "adaptive", "display": "summarized"},
        output_config={"effort": EFFORT},
        tools=tools,
        messages=[
            {
                "role": "user",
                "content": "I run an outdoor tech meetup in London tomorrow evening and I'm deciding whether to move it indoors and what to open the talk with. Work out which of your tools you need and in what order, use them, then give me a recommendation with your reasoning.",
            }
        ],
    )

    # Print detailed information about initial response
    print("\n=== INITIAL RESPONSE ===")
    print(f"Response ID: {response.id}")
    print(f"Stop reason: {response.stop_reason}")
    print(f"Model: {response.model}")
    print(f"Content blocks: {len(response.content)} blocks")

    # Print each content block
    for i, block in enumerate(response.content):
        print(f"\nBlock {i + 1}: Type = {block.type}")
        if block.type == "thinking":
            print(f"Thinking content: {block.thinking[:150]}...")
            print(f"Signature available: {bool(getattr(block, 'signature', None))}")
        elif block.type == "text":
            print(f"Text content: {block.text}")
        elif block.type == "tool_use":
            print(f"Tool: {block.name}")
            print(f"Tool input: {block.input}")
            print(f"Tool ID: {block.id}")
    print("=== END INITIAL RESPONSE ===\n")

    # Handle potentially multiple tool calls
    full_conversation = [
        {
            "role": "user",
            "content": "I run an outdoor tech meetup in London tomorrow evening and I'm deciding whether to move it indoors and what to open the talk with. Work out which of your tools you need and in what order, use them, then give me a recommendation with your reasoning.",
        }
    ]

    # Track iteration count for multi-turn tool use
    iteration = 0

    while response.stop_reason == "tool_use":
        iteration += 1
        print(f"\n=== TOOL USE ITERATION {iteration} ===")

        # Keep the whole assistant turn (thinking, text and tool_use blocks, in order)
        # for the conversation history; pass it back exactly as received
        assistant_blocks = response.content

        # Add assistant response with thinking blocks and tool use
        full_conversation.append({"role": "assistant", "content": assistant_blocks})

        # Run every tool_use block in this turn (Claude may call several tools in parallel)
        tool_use_blocks = [block for block in response.content if block.type == "tool_use"]
        if tool_use_blocks:
            tool_results = []
            for tool_use_block in tool_use_blocks:
                print("\n=== EXECUTING TOOL ===")
                print(f"Tool name: {tool_use_block.name}")

                # Execute the appropriate tool
                if tool_use_block.name == "weather":
                    print(f"Location to check: {tool_use_block.input['location']}")
                    tool_result = weather(tool_use_block.input["location"])
                elif tool_use_block.name == "news":
                    print(f"Topic to check: {tool_use_block.input['topic']}")
                    tool_result = news(tool_use_block.input["topic"])
                else:
                    tool_result = {"error": "Unknown tool"}

                print(f"Result: {tool_result}")
                print("=== TOOL EXECUTION COMPLETE ===\n")
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_use_block.id,
                        "content": json.dumps(tool_result),
                    }
                )

            # Add all tool results to the conversation in a single user turn
            full_conversation.append({"role": "user", "content": tool_results})

            # Continue the conversation
            print("\n=== SENDING FOLLOW-UP REQUEST WITH TOOL RESULT ===")
            response = client.messages.create(
                model=MODEL_NAME,
                max_tokens=MAX_TOKENS,
                thinking={"type": "adaptive", "display": "summarized"},
                output_config={"effort": EFFORT},
                tools=tools,
                messages=full_conversation,
            )

            # Print follow-up response details
            print(f"\n=== FOLLOW-UP RESPONSE (ITERATION {iteration}) ===")
            print(f"Response ID: {response.id}")
            print(f"Stop reason: {response.stop_reason}")
            print(f"Content blocks: {len(response.content)} blocks")

            for i, block in enumerate(response.content):
                print(f"\nBlock {i + 1}: Type = {block.type}")
                if block.type == "thinking":
                    print(f"Thinking content preview: {block.thinking[:100]}...")
                elif block.type == "text":
                    print(f"Text content preview: {block.text[:100]}...")
                elif block.type == "tool_use":
                    print(f"Tool: {block.name}")
                    print(f"Tool input preview: {str(block.input)[:100]}")
            print(f"=== END FOLLOW-UP RESPONSE (ITERATION {iteration}) ===\n")

            if response.stop_reason != "tool_use":
                print("\n=== FINAL RESPONSE ===")
                print_thinking_response(response)
                print("=== END FINAL RESPONSE ===")
        else:
            print("No tool_use block found in response.")
            break


# Run the example
multiple_tool_calls_with_thinking()


=== INITIAL RESPONSE ===
Response ID: msg_011Cfdj62iENiRCj8mRZbo5x
Stop reason: tool_use
Model: claude-sonnet-5-5
Content blocks: 4 blocks

Block 1: Type = thinking
Thinking content: London weather and outdoor tech news are unrelated requests, so I can fetch both at once.

...
Signature available: True

Block 2: Type = text
Text content: I'll check two things: the London weather, which decides indoors or outdoors, and current outdoor tech news, which could give you an opening for the talk. Neither depends on the other, so I'll run both now.

Block 3: Type = tool_use
Tool: weather
Tool input: {'location': 'London'}
Tool ID: toolu_016w2JwNyoLzMzEVbLpH62xJ

Block 4: Type = tool_use
Tool: news
Tool input: {'topic': 'outdoor tech'}
Tool ID: toolu_01VbrLTn6hoqbSCzFdsJSTYm
=== END INITIAL RESPONSE ===


=== TOOL USE ITERATION 1 ===

=== EXECUTING TOOL ===
Tool name: weather
Location to check: London
Result: {'temperature': 62, 'condition': 'Cloudy'}
=== TOOL EXECUTION COMPLETE ===


=== EXEC

## Preserving thinking blocks

When working with extended thinking and tools, make sure to:

1. **Pass thinking blocks back unchanged, with their signatures.** Each `thinking` block carries a cryptographic `signature` over the full (unsummarized) reasoning. When you send tool results, include the previous assistant turn exactly as you received it: `thinking` / `redacted_thinking` / `text` / `tool_use` blocks, in order. A block whose signature doesn't verify is rejected with a 400.

2. **Don't edit earlier turns.** On Claude Sonnet 5.5 a thinking block is also bound to the conversation prefix that produced it (the system prompt, the `tools` array, and every earlier message). If you replay a thinking block after changing any of that, the block no longer matches its conversation. Accounts created on or after August 31, 2026 get a 400 for this by default; older accounts can opt in with the `thinking-binding-controls-2026-08-01` beta header and `thinking.block_binding.prefix_mismatch_behavior` set to `"error"` (reject the request) or `"drop_block"` (drop the stale block and report it in `input_transformations`). The example below uses the beta header so the behavior is the same on any account.

3. **Don't silently drop them either.** Leaving the thinking blocks out of the replayed turn is accepted by the API, but Claude then continues the task without the reasoning it did before the tool call. Keep them.

4. **Handle both `thinking` and `redacted_thinking` blocks.** Both must be preserved in the conversation history, even though the content of redacted blocks is not human readable.

For more details on extended thinking without tools, see the main "Extended Thinking" notebook.

In [4]:
def thinking_block_preservation_example():
    # Define a simple weather tool
    tools = [
        {
            "name": "weather",
            "description": "Get current weather information for a location.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The location to get weather for.",
                    }
                },
                "required": ["location"],
            },
        }
    ]

    def weather(location):
        # Mock weather data, keyed by bare city name (the model may pass "Paris, FR")
        location = location.split(",")[0].strip()
        weather_data = {
            "New York": {"temperature": 72, "condition": "Sunny"},
            "London": {"temperature": 62, "condition": "Cloudy"},
            "Tokyo": {"temperature": 80, "condition": "Partly cloudy"},
            "Paris": {"temperature": 65, "condition": "Rainy"},
            "Sydney": {"temperature": 85, "condition": "Clear"},
            "Berlin": {"temperature": 60, "condition": "Foggy"},
        }

        return weather_data.get(location, {"error": f"No weather data available for {location}"})

    question = "I'm about to cycle across Berlin to a meeting I can't be late for. Work out what you need to check, use the tool, then tell me whether to leave extra time and why."

    # Opt in to strict prefix binding so this demo behaves the same on every account
    # (accounts created on or after 2026-08-31 get this behavior by default).
    betas = ["thinking-binding-controls-2026-08-01"]
    strict_thinking = {
        "type": "adaptive",
        "display": "summarized",
        "block_binding": {"prefix_mismatch_behavior": "error"},
    }

    # Initial request with tool use and thinking. Adaptive thinking occasionally skips
    # thinking even here, so retry a couple of times until we have a thinking block to work with.
    for _ in range(3):
        response = client.beta.messages.create(
            model=MODEL_NAME,
            max_tokens=MAX_TOKENS,
            thinking=strict_thinking,
            output_config={"effort": EFFORT},
            tools=tools,
            betas=betas,
            messages=[{"role": "user", "content": question}],
        )
        thinking_blocks = [b for b in response.content if b.type == "thinking"]
        tool_use_blocks = [b for b in response.content if b.type == "tool_use"]
        if thinking_blocks and tool_use_blocks:
            break

    print("\n=== INITIAL RESPONSE ===")
    print("Response contains:")
    print(f"- {len(thinking_blocks)} thinking blocks")
    print(f"- {len(tool_use_blocks)} tool use blocks")

    if not (thinking_blocks and tool_use_blocks):
        print("\nClaude answered without a thinking block or without calling the tool this time;")
        print("re-run the cell to see the preservation checks below.")
        return

    tool_block = tool_use_blocks[0]
    print(f"\nTool called: {tool_block.name}")
    print(f"Location to check: {tool_block.input['location']}")

    # Execute the tool
    tool_result = weather(tool_block.input["location"])
    print(f"Tool result: {tool_result}")
    tool_result_turn = {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_block.id,
                "content": json.dumps(tool_result),
            }
        ],
    }

    def follow_up(label, messages, thinking=strict_thinking):
        try:
            result = client.beta.messages.create(
                model=MODEL_NAME,
                max_tokens=MAX_TOKENS,
                thinking=thinking,
                output_config={"effort": EFFORT},
                tools=tools,
                betas=betas,
                messages=messages,
            )
            print(f"SUCCESS ({label}): stop_reason={result.stop_reason}")
            return result
        except anthropic.BadRequestError as e:
            print(f"ERROR ({label}): {e}")
            return None

    # TEST 1: tamper with the thinking block's signature -> rejected
    print("\n=== TEST 1: modified signature ===")
    tampered_turn = []
    for block in response.content:
        block_dict = block.model_dump()
        if block_dict["type"] == "thinking":
            block_dict["signature"] = block_dict["signature"][:-4] + "AAAA"
        tampered_turn.append(block_dict)
    follow_up(
        "tampered signature",
        [
            {"role": "user", "content": question},
            {"role": "assistant", "content": tampered_turn},
            tool_result_turn,
        ],
    )
    print("Thinking blocks must be passed back byte-for-byte, signature included.")

    # TEST 2: replay the thinking block unchanged but edit the earlier user message.
    # The block is bound to the original prefix, so with prefix_mismatch_behavior="error"
    # the request is rejected...
    print("\n=== TEST 2: edited earlier turn, prefix_mismatch_behavior='error' ===")
    edited_history = [
        {"role": "user", "content": question + " Thanks!"},  # <- not what Claude saw
        {"role": "assistant", "content": response.content},
        tool_result_turn,
    ]
    follow_up("edited prefix", edited_history)

    # ...and with "drop_block" the API drops the stale thinking block, tells you it did so
    # in input_transformations, and Claude continues without that reasoning.
    print("\n=== TEST 3: edited earlier turn, prefix_mismatch_behavior='drop_block' ===")
    lenient = {**strict_thinking, "block_binding": {"prefix_mismatch_behavior": "drop_block"}}
    dropped = follow_up("edited prefix, drop_block", edited_history, thinking=lenient)
    if dropped is not None:
        print(f"input_transformations: {dropped.to_dict().get('input_transformations')}")

    # TEST 4: the correct approach - every block from the first response, in order, unchanged,
    # on top of an unchanged prefix.
    print("\n=== TEST 4: WITH thinking block, unchanged history (correct approach) ===")
    complete_response = follow_up(
        "full replay",
        [
            {"role": "user", "content": question},
            {"role": "assistant", "content": response.content},
            tool_result_turn,
        ],
    )
    if complete_response is not None:
        second_thinking = [b for b in complete_response.content if b.type == "thinking"]
        second_text = [b for b in complete_response.content if b.type == "text"]
        print("\nSecond response contains:")
        print(f"- {len(second_thinking)} thinking blocks")
        print(f"- {len(second_text)} text blocks")
        if second_text:
            print(f"\nFinal answer: {second_text[0].text}")
        print(
            "\nNote: with adaptive thinking Claude may or may not think again after a tool result,"
        )
        print(
            "so the second response can contain thinking blocks. Either way, pass every earlier block back unchanged."
        )


# Run the example
thinking_block_preservation_example()


=== INITIAL RESPONSE ===
Response contains:
- 1 thinking blocks
- 1 tool use blocks

Tool called: weather
Location to check: Berlin
Tool result: {'temperature': 60, 'condition': 'Foggy'}

=== TEST 1: modified signature ===
ERROR (tampered signature): Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.1.content.0: Invalid `signature` in `thinking` block'}, 'request_id': 'req_011Cfdj789gkiDGRiGuVqr8n'}
Thinking blocks must be passed back byte-for-byte, signature included.

=== TEST 2: edited earlier turn, prefix_mismatch_behavior='error' ===
ERROR (edited prefix): Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.1.content.0: Invalid `signature` in `thinking` block. The block is bound to a different conversation. Remove the block, or set `thinking.block_binding.prefix_mismatch_behavior` to "drop_block". Content before this block differs from when it was created, first at `messages.0.conten

## Conclusion

This notebook shows how to combine Claude's extended thinking feature with tool use. Key benefits include:

1. Transparency into Claude's thinking process when using tools
2. Visibility into how Claude decides when to use tools versus internal knowledge
3. Better understanding of multi-step tasks that involve multiple tool calls
4. Insight into how Claude interprets tool results and incorporates them into responses

When using extended thinking with tools, keep in mind:
- Pick an `effort` level that fits the task (`"high"` is the default; `"low"` and `"medium"` trade depth for speed and cost, `"xhigh"` and `"max"` are for the hardest problems). There is no token budget on current models
- Set `"display": "summarized"` if you want to read the thinking; the default (`"omitted"`) returns empty thinking text
- Always preserve thinking blocks and their signatures when passing tool results
- Include both normal and redacted thinking blocks in the conversation history
- Ensure that system prompts, tools, and thinking configurations match between calls
- Claude may think again between tool calls (interleaved thinking), so handle `thinking` blocks in any assistant turn and pass them all back unchanged
- Tool use and thinking together can increase token usage and response time